In [1]:
import pandas as pd

In [2]:
def read_data(region, year):
    if region.upper() == "ERCOT":
        milp_df = pd.read_csv(f"MILP_forecast_RF_one_year_{region.upper()}_{year}.csv")
    else:
        milp_df = pd.read_csv(f"MILP_forecast_RF_one_year_{region.upper()}_{year}.csv")
    rl_df = pd.read_csv(f"rlPPO_forecast_RF_one_year_{region}_{year}.csv")
    heuristictime_df = pd.read_csv(f"HeurTime_forecast_RF_one_year_{region.upper()}_{year}.csv")
    heuristicquantile_df = pd.read_csv(f"HeurQuantile_forecast_RF_one_year_{region.upper()}_{year}.csv")
    return rl_df, milp_df, heuristictime_df, heuristicquantile_df

In [3]:
def perform(df):
    # Recompute hourly profit (or use existing profit_step column)
    df["profit_step"] = (df["discharge_MW"] - df["charge_MW"]) * df["prices_actual"] * 1

    # Assign a day index (assumes rows are ordered hourly, 24 rows per day)
    df["day"] = df.index // 24

    # Sum hourly profits within each day → daily profit ($/day)
    daily_profit = df.groupby("day")["profit_step"].sum()

    # Summary stats
    mean_profit = daily_profit.mean()
    std_profit  = daily_profit.std()

    # print(f"Mean daily profit : ${mean_profit:,.2f}/day")
    # print(f"Std dev           : ${std_profit:,.2f}/day")
    print(f"Mean Daily Report: ${mean_profit:,.2f} ± ${std_profit:,.2f} $/day")

    median_profit = daily_profit.median()
    q25, q75 = daily_profit.quantile(0.25), daily_profit.quantile(0.75)
    print(f"Median Daily Report${median_profit:,.0f} [{q25:,.0f}, {q75:,.0f}]")

    sharpe_daily = daily_profit.mean() / daily_profit.std()
    sharpe_annualised = sharpe_daily * (365 ** 0.5)
    print(f"Sharpe Daily Report: {sharpe_daily:.2f}")
    print(f"Sharpe Annualised Report: {sharpe_annualised:.2f} \n")

    # 8,760 hourly profit observations
    hourly_profit = df["profit_step"]  # already per hour
    sharpe_hourly = hourly_profit.mean() / hourly_profit.std()
    sharpe_annualised = sharpe_hourly * (8760 ** 0.5)
    # print(f"Mean hourly profit : ${hourly_profit.mean():,.2f}/hour")
    # print(f"Std dev            : ${hourly_profit.std():,.2f}/hour")
    print(f"Mean Hourly Report: ${hourly_profit.mean():,.2f} ± ${hourly_profit.std():,.2f} $/hour")
    median_hourly = hourly_profit.median()
    q25_hourly, q75_hourly = hourly_profit.quantile(0.25), hourly_profit.quantile(0.75)
    print(f"Median Hourly Report: ${median_hourly:,.2f} [{q25_hourly:,.2f}, {q75_hourly:,.2f}] $/hour")
    print(f"Sharpe ratio (hourly): {sharpe_hourly:.2f}")
    print(f"Sharpe ratio (annualised): {sharpe_annualised:.2f} \n")

    # print cumulative profit over the year
    cumulative_profit = daily_profit.sum()
    print(f"Cumulative profit over the year: ${cumulative_profit:,.2f}")



In [4]:
REGION="Ercot"
YEAR1,YEAR2=2022,2023
rl_df, milp_df, heuristictime_df, heuristicquantile_df = read_data(REGION, YEAR1)

In [5]:
perform(rl_df)

Mean Daily Report: $2,754.50 ± $2,910.20 $/day
Median Daily Report$2,185 [1,315, 3,177]
Sharpe Daily Report: 0.95
Sharpe Annualised Report: 18.08 

Mean Hourly Report: $114.77 ± $527.78 $/hour
Median Hourly Report: $0.00 [-101.58, 218.64] $/hour
Sharpe ratio (hourly): 0.22
Sharpe ratio (annualised): 20.35 

Cumulative profit over the year: $999,884.23


In [6]:
perform(milp_df)

Mean Daily Report: $977.36 ± $5,040.35 $/day
Median Daily Report$0 [0, 0]
Sharpe Daily Report: 0.19
Sharpe Annualised Report: 3.70 

Mean Hourly Report: $40.72 ± $571.73 $/hour
Median Hourly Report: $0.00 [0.00, 0.00] $/hour
Sharpe ratio (hourly): 0.07
Sharpe ratio (annualised): 6.67 

Cumulative profit over the year: $354,780.41


In [7]:
perform(heuristictime_df)

Mean Daily Report: $1,354.49 ± $2,231.35 $/day
Median Daily Report$1,140 [672, 1,789]
Sharpe Daily Report: 0.61
Sharpe Annualised Report: 11.60 

Mean Hourly Report: $56.44 ± $972.37 $/hour
Median Hourly Report: $9.07 [-429.31, 492.60] $/hour
Sharpe ratio (hourly): 0.06
Sharpe ratio (annualised): 5.43 

Cumulative profit over the year: $491,679.89


In [8]:
perform(heuristicquantile_df)

Mean Daily Report: $2,249.59 ± $4,305.41 $/day
Median Daily Report$1,260 [522, 2,465]
Sharpe Daily Report: 0.52
Sharpe Annualised Report: 9.98 

Mean Hourly Report: $93.73 ± $697.07 $/hour
Median Hourly Report: $0.00 [0.00, 0.00] $/hour
Sharpe ratio (hourly): 0.13
Sharpe ratio (annualised): 12.59 

Cumulative profit over the year: $816,600.43


In [9]:
rl_df, milp_df, heuristictime_df, heuristicquantile_df = read_data(REGION, YEAR2)

In [10]:
perform(rl_df)

Mean Daily Report: $2,374.84 ± $6,277.76 $/day
Median Daily Report$1,013 [648, 1,676]
Sharpe Daily Report: 0.38
Sharpe Annualised Report: 7.23 

Mean Hourly Report: $98.95 ± $765.55 $/hour
Median Hourly Report: $0.00 [-40.99, 69.28] $/hour
Sharpe ratio (hourly): 0.13
Sharpe ratio (annualised): 12.10 

Cumulative profit over the year: $745,701.12


In [11]:
perform(milp_df)

Mean Daily Report: $85.51 ± $459.47 $/day
Median Daily Report$0 [0, 0]
Sharpe Daily Report: 0.19
Sharpe Annualised Report: 3.56 

Mean Hourly Report: $3.56 ± $93.16 $/hour
Median Hourly Report: $0.00 [0.00, 0.00] $/hour
Sharpe ratio (hourly): 0.04
Sharpe ratio (annualised): 3.58 

Cumulative profit over the year: $26,849.05


In [12]:
perform(heuristictime_df)

Mean Daily Report: $2,050.37 ± $6,074.55 $/day
Median Daily Report$880 [551, 1,443]
Sharpe Daily Report: 0.34
Sharpe Annualised Report: 6.45 

Mean Hourly Report: $85.43 ± $944.32 $/hour
Median Hourly Report: $6.60 [-202.23, 220.00] $/hour
Sharpe ratio (hourly): 0.09
Sharpe ratio (annualised): 8.47 

Cumulative profit over the year: $643,816.74


In [13]:
perform(heuristicquantile_df)

Mean Daily Report: $1,694.63 ± $3,593.00 $/day
Median Daily Report$882 [471, 1,473]
Sharpe Daily Report: 0.47
Sharpe Annualised Report: 9.01 

Mean Hourly Report: $70.61 ± $497.32 $/hour
Median Hourly Report: $0.00 [0.00, 0.00] $/hour
Sharpe ratio (hourly): 0.14
Sharpe ratio (annualised): 13.29 

Cumulative profit over the year: $532,113.24


In [14]:
REGION="NewYork"
rl_df, milp_df, heuristictime_df, heuristicquantile_df = read_data(REGION, YEAR1)

In [15]:
perform(rl_df)

Mean Daily Report: $2,683.76 ± $3,271.49 $/day
Median Daily Report$1,913 [1,054, 3,238]
Sharpe Daily Report: 0.82
Sharpe Annualised Report: 15.67 

Mean Hourly Report: $111.82 ± $689.68 $/hour
Median Hourly Report: $0.00 [0.00, 123.02] $/hour
Sharpe ratio (hourly): 0.16
Sharpe ratio (annualised): 15.18 

Cumulative profit over the year: $976,889.61


In [16]:
perform(milp_df)

Mean Daily Report: $2,829.75 ± $4,845.10 $/day
Median Daily Report$1,819 [1,149, 3,362]
Sharpe Daily Report: 0.58
Sharpe Annualised Report: 11.16 

Mean Hourly Report: $117.91 ± $1,009.04 $/hour
Median Hourly Report: $0.00 [-63.18, 405.58] $/hour
Sharpe ratio (hourly): 0.12
Sharpe ratio (annualised): 10.94 

Cumulative profit over the year: $1,030,027.90


In [17]:
perform(heuristictime_df)

Mean Daily Report: $1,682.20 ± $3,850.90 $/day
Median Daily Report$874 [364, 1,845]
Sharpe Daily Report: 0.44
Sharpe Annualised Report: 8.35 

Mean Hourly Report: $70.09 ± $1,142.78 $/hour
Median Hourly Report: $13.75 [-558.89, 612.32] $/hour
Sharpe ratio (hourly): 0.06
Sharpe ratio (annualised): 5.74 

Cumulative profit over the year: $612,320.84


In [18]:
perform(heuristicquantile_df)

Mean Daily Report: $1,814.65 ± $4,343.00 $/day
Median Daily Report$1,111 [334, 2,255]
Sharpe Daily Report: 0.42
Sharpe Annualised Report: 7.98 

Mean Hourly Report: $75.61 ± $828.96 $/hour
Median Hourly Report: $0.00 [0.00, 0.00] $/hour
Sharpe ratio (hourly): 0.09
Sharpe ratio (annualised): 8.54 

Cumulative profit over the year: $660,532.47


In [19]:
rl_df, milp_df, heuristictime_df, heuristicquantile_df = read_data(REGION, YEAR2)

In [20]:
perform(rl_df)

Mean Daily Report: $1,009.48 ± $1,442.23 $/day
Median Daily Report$661 [388, 1,053]
Sharpe Daily Report: 0.70
Sharpe Annualised Report: 13.37 

Mean Hourly Report: $42.06 ± $300.17 $/hour
Median Hourly Report: $0.00 [-14.48, 52.73] $/hour
Sharpe ratio (hourly): 0.14
Sharpe ratio (annualised): 13.12 

Cumulative profit over the year: $367,450.88


In [21]:
perform(milp_df)

Mean Daily Report: $984.20 ± $1,702.34 $/day
Median Daily Report$697 [364, 1,095]
Sharpe Daily Report: 0.58
Sharpe Annualised Report: 11.05 

Mean Hourly Report: $41.01 ± $384.20 $/hour
Median Hourly Report: $0.00 [0.00, 0.00] $/hour
Sharpe ratio (hourly): 0.11
Sharpe ratio (annualised): 9.99 

Cumulative profit over the year: $358,250.37


In [22]:
perform(heuristictime_df)

Mean Daily Report: $705.08 ± $1,457.56 $/day
Median Daily Report$451 [195, 726]
Sharpe Daily Report: 0.48
Sharpe Annualised Report: 9.24 

Mean Hourly Report: $29.38 ± $440.98 $/hour
Median Hourly Report: $0.00 [-253.03, 264.25] $/hour
Sharpe ratio (hourly): 0.07
Sharpe ratio (annualised): 6.24 

Cumulative profit over the year: $256,648.95


In [23]:
perform(heuristicquantile_df)

Mean Daily Report: $436.75 ± $1,582.72 $/day
Median Daily Report$219 [-112, 658]
Sharpe Daily Report: 0.28
Sharpe Annualised Report: 5.27 

Mean Hourly Report: $18.20 ± $307.10 $/hour
Median Hourly Report: $0.00 [0.00, 0.00] $/hour
Sharpe ratio (hourly): 0.06
Sharpe ratio (annualised): 5.55 

Cumulative profit over the year: $158,978.40


In [24]:
gpt4_ny2022= pd.read_csv(f"gpt4_hourly_arbitrage_new_york_2022.csv")
gpt4_ny2023= pd.read_csv(f"gpt4_hourly_arbitrage_new_york_2023.csv")
gpt4_er2022= pd.read_csv(f"gpt4_hourly_arbitrage_texas_2022.csv")
gpt4_er2023= pd.read_csv(f"gpt4_hourly_arbitrage_texas_2023.csv")

In [26]:
perform(gpt4_er2022)

Mean Daily Report: $1,290.38 ± $4,105.38 $/day
Median Daily Report$844 [319, 1,341]
Sharpe Daily Report: 0.31
Sharpe Annualised Report: 6.00 

Mean Hourly Report: $53.77 ± $543.57 $/hour
Median Hourly Report: $0.00 [0.00, 113.08] $/hour
Sharpe ratio (hourly): 0.10
Sharpe ratio (annualised): 9.26 

Cumulative profit over the year: $468,406.54


In [28]:
perform(gpt4_er2023)

Mean Daily Report: $627.78 ± $1,369.71 $/day
Median Daily Report$457 [239, 759]
Sharpe Daily Report: 0.46
Sharpe Annualised Report: 8.76 

Mean Hourly Report: $26.16 ± $285.06 $/hour
Median Hourly Report: $0.00 [0.00, 74.63] $/hour
Sharpe ratio (hourly): 0.09
Sharpe ratio (annualised): 8.59 

Cumulative profit over the year: $197,121.76


In [25]:
perform(gpt4_ny2022)

Mean Daily Report: $2,201.79 ± $4,754.62 $/day
Median Daily Report$1,472 [863, 2,521]
Sharpe Daily Report: 0.46
Sharpe Annualised Report: 8.85 

Mean Hourly Report: $91.74 ± $900.50 $/hour
Median Hourly Report: $0.00 [0.00, 303.17] $/hour
Sharpe ratio (hourly): 0.10
Sharpe ratio (annualised): 9.54 

Cumulative profit over the year: $810,259.08


In [27]:
perform(gpt4_ny2023)

Mean Daily Report: $836.22 ± $1,277.14 $/day
Median Daily Report$664 [358, 1,040]
Sharpe Daily Report: 0.65
Sharpe Annualised Report: 12.51 

Mean Hourly Report: $34.84 ± $307.61 $/hour
Median Hourly Report: $0.00 [0.00, 117.18] $/hour
Sharpe ratio (hourly): 0.11
Sharpe ratio (annualised): 10.60 

Cumulative profit over the year: $304,382.98


In [5]:

italy_milp = pd.read_csv(f"MILP_forecast_RF_one_year.csv")
italy_heuristicquantile = pd.read_csv(f"HeurQuantile_forecast_RF_one_year.csv")
italy_heuristictime = pd.read_csv(f"HeurTime_forecast_RF_one_year.csv")
italy_gpt4o = pd.read_csv(f"gpt4_hourly_arbitrage_corrected3.csv")
italy_rl = pd.read_csv(f"rlPPO_forecast_RF_one_year.csv")

italy_milp.shape, italy_heuristicquantile.shape, italy_heuristictime.shape, italy_gpt4o.shape, italy_rl.shape

((8736, 12), (8736, 10), (8736, 10), (8736, 10), (8736, 10))

In [6]:
perform(italy_milp)

Mean Daily Report: $1,747.33 ± $364.66 $/day
Median Daily Report$1,712 [1,471, 1,993]
Sharpe Daily Report: 4.79
Sharpe Annualised Report: 91.54 

Mean Hourly Report: $72.81 ± $394.94 $/hour
Median Hourly Report: $0.00 [0.00, 420.49] $/hour
Sharpe ratio (hourly): 0.18
Sharpe ratio (annualised): 17.25 

Cumulative profit over the year: $636,026.92


In [9]:
perform(italy_gpt4o)

Mean Daily Report: $1,670.20 ± $576.52 $/day
Median Daily Report$1,689 [1,384, 2,071]
Sharpe Daily Report: 2.90
Sharpe Annualised Report: 55.35 

Mean Hourly Report: $69.59 ± $437.33 $/hour
Median Hourly Report: $0.00 [0.00, 444.98] $/hour
Sharpe ratio (hourly): 0.16
Sharpe ratio (annualised): 14.89 

Cumulative profit over the year: $607,951.33


In [10]:
perform(italy_rl)

Mean Daily Report: $1,558.26 ± $396.30 $/day
Median Daily Report$1,506 [1,276, 1,853]
Sharpe Daily Report: 3.93
Sharpe Annualised Report: 75.12 

Mean Hourly Report: $64.93 ± $319.13 $/hour
Median Hourly Report: $0.00 [-45.82, 212.91] $/hour
Sharpe ratio (hourly): 0.20
Sharpe ratio (annualised): 19.04 

Cumulative profit over the year: $567,208.37


In [8]:
perform(italy_heuristictime)

Mean Daily Report: $1,234.22 ± $340.08 $/day
Median Daily Report$1,208 [992, 1,480]
Sharpe Daily Report: 3.63
Sharpe Annualised Report: 69.34 

Mean Hourly Report: $51.43 ± $532.50 $/hour
Median Hourly Report: $11.16 [-490.17, 543.79] $/hour
Sharpe ratio (hourly): 0.10
Sharpe ratio (annualised): 9.04 

Cumulative profit over the year: $449,255.14


In [7]:
perform(italy_heuristicquantile)

Mean Daily Report: $1,272.02 ± $533.43 $/day
Median Daily Report$1,295 [913, 1,637]
Sharpe Daily Report: 2.38
Sharpe Annualised Report: 45.56 

Mean Hourly Report: $53.00 ± $336.06 $/hour
Median Hourly Report: $0.00 [0.00, 0.00] $/hour
Sharpe ratio (hourly): 0.16
Sharpe ratio (annualised): 14.76 

Cumulative profit over the year: $463,014.12
